# Model Validation Against Manual Labels

This notebook compares manual Label Studio labels against the saved CNN.

Pipeline:
1. Load Label Studio export (`results/*.csv`)
2. Reconstruct matching ERP matrices from `data_fixations.hdf5`
3. Preprocess to `64x64` using the same helper logic as in `data_vis.ipynb`
4. Feed tensor `(H, W, C, N)` into the model
5. Compare model predictions vs. manual labels


In [1]:
versioninfo()

Julia Version 1.12.3
Commit 966d0af0fdf (2025-12-15 11:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 16 × AMD Ryzen 7 7800X3D 8-Core Processor
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, znver4)
  GC: Built with stock GC
Threads: 16 default, 1 interactive, 16 GC (on 16 virtual cores)
Environment:
  JULIA_NUM_THREADS = 16


In [2]:
import Pkg

# Make notebook startup more stable in VS Code notebooks:
# - disable automatic background precompile
# - limit precompile workers to 1
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

Pkg.activate(".")

# Do NOT run Pkg.add(...) automatically in notebooks.
# If PrettyTables is missing, install once manually:
#   import Pkg; Pkg.activate("."); Pkg.add("PrettyTables")

using CSV, DataFrames, HDF5, JLD2, Statistics, Printf, Dates
using ImageFiltering: imfilter
using Images: imresize
using PrettyTables

# Flux lives in the model_train environment; extend LOAD_PATH instead of changing Project.toml
push!(LOAD_PATH, abspath(joinpath(pwd(), "..", "model_train")))
using Flux
using Flux: onecold



  Activating project at `~/Dokumente/BA2/notebooks/model_test`


In [3]:
results_csv_paths = [
    joinpath(pwd(), "results", "project-14-at-2026-02-15-23-09-f5225e5c.csv"),
    joinpath(pwd(), "results", "project-15-at-2026-02-18-19-35-828515fe.csv"),
]

model_path = joinpath(pwd(), "cnn_model_20260208_014145.jld2")
h5_path = joinpath(pwd(), "data_fixations.hdf5")
events_csv_path = joinpath(pwd(), "events.csv")

for p in vcat(results_csv_paths, [model_path, h5_path, events_csv_path])
    @assert isfile(p) "File not found: $p"
end

sampling_rate = 512
pre_stim_s = 0.5
pre_samples = Int(round(pre_stim_s * sampling_rate))
time_zero_idx = pre_samples + 1

target_height = 64
target_width = 64
resize_to_model_input = true

println("results_csv_paths =")
for p in results_csv_paths
    println("  - ", p)
end
println("model_path       = ", model_path)
println("resize_to_model_input = ", resize_to_model_input)
println("preprocessing source = manual_labeling_prepare.ipynb (same order + same constants)")



results_csv_paths =
  - /home/benjamin/Dokumente/BA2/notebooks/model_test/results/project-14-at-2026-02-15-23-09-f5225e5c.csv
  - /home/benjamin/Dokumente/BA2/notebooks/model_test/results/project-15-at-2026-02-18-19-35-828515fe.csv
model_path       = /home/benjamin/Dokumente/BA2/notebooks/model_test/cnn_model_20260208_014145.jld2
resize_to_model_input = true
preprocessing source = manual_labeling_prepare.ipynb (same order + same constants)


In [4]:
function load_erps_from_h5(path::AbstractString)
    return h5open(path, "r") do f
        candidates = ["erps", "/erps", "data", "/data/data_fixations.hdf5", "data/data_fixations.hdf5"]
        for key in candidates
            if haskey(f, key)
                obj = f[key]
                if obj isa HDF5.Dataset
                    return read(obj)
                end
            end
        end
        # Fallback: first dataset found recursively
        function first_dataset(g)
            for k in keys(g)
                obj = g[k]
                if obj isa HDF5.Dataset
                    return read(obj)
                elseif obj isa HDF5.Group
                    x = first_dataset(obj)
                    x === nothing || return x
                end
            end
            return nothing
        end
        x = first_dataset(f)
        x === nothing && error("No dataset found in HDF5 file: $path")
        return x
    end
end

function load_and_merge_label_sources(paths::Vector{String})
    dfs = DataFrame[]
    for p in paths
        df = CSV.read(p, DataFrame)
        df.source_csv = fill(basename(p), nrow(df))
        push!(dfs, df)
    end

    labels_all = vcat(dfs...; cols = :union)

    if :image in names(labels_all)
        updated_at_str = :updated_at in names(labels_all) ? string.(coalesce.(labels_all.updated_at, "")) : fill("", nrow(labels_all))
        created_at_str = :created_at in names(labels_all) ? string.(coalesce.(labels_all.created_at, "")) : fill("", nrow(labels_all))
        labels_all.updated_at_str = updated_at_str
        labels_all.created_at_str = created_at_str

        # Keep newest annotation per image (ISO timestamps sort lexicographically)
        sort!(labels_all, [:image, :updated_at_str, :created_at_str], rev = [false, true, true])
        labels_merged = unique(labels_all, :image)
    else
        labels_merged = labels_all
    end

    return labels_all, labels_merged
end

erps = load_erps_from_h5(h5_path)
events = CSV.read(events_csv_path, DataFrame)
labels_all_df, labels_df = load_and_merge_label_sources(results_csv_paths)

model_bundle = JLD2.load(model_path)
model = cpu(model_bundle["model"])
model_pattern_names = Symbol.(model_bundle["pattern_names"])

println("ERP shape: ", size(erps))
println("Events rows: ", nrow(events))
println("Label rows (all imported): ", nrow(labels_all_df))
println("Label rows (merged unique by image): ", nrow(labels_df))
if :source_csv in names(labels_all_df)
    source_counts = combine(groupby(labels_all_df, :source_csv), nrow => :count)
    sort!(source_counts, :source_csv)
    println("Rows per source file:")
    show(source_counts, allrows = true, allcols = true)
    println()
end
println("Model classes: ", model_pattern_names)
println("Model expects 64x64 and binary labels (:no_pattern/:pattern).")



ERP shape: (128, 769, 2508)
Events rows: 2508
Label rows (all imported): 500
Label rows (merged unique by image): 500
Model classes: [:no_pattern, :pattern]
Model expects 64x64 and binary labels (:no_pattern/:pattern).


In [5]:
# Helper functions copied from manual_labeling_prepare.ipynb
const FILTER_BORDER = "reflect"
const LOWPASS_SIGMA = 75.0f0
const LOWPASS_KERNEL_SIZE = (21, 21)

include(joinpath(pwd(), "..", "utils", "erp_image_utils.jl"))
using .ERPImageUtils: gaussian_kernel, zscore_timepoints

function sortvalues_from(df::DataFrame, col::Symbol)
    v = df[!, col]
    if eltype(v) <: Number
        return Float64.(v)
    end
    return collect(v)
end

function process_erp_timed(erps, events::DataFrame, channel::Int, sort_col::Symbol, time_zero_idx::Int;
                           low_pass_factor::Real = LOWPASS_SIGMA)
    @assert 1 <= channel <= size(erps, 1) "channel out of range"
    @assert sort_col in propertynames(events) "sort column not found: $(sort_col)"

    # 1) Extract post-stimulus matrix (time x trials)
    t_extract0 = time_ns()
    data = Float32.(erps[channel, time_zero_idx:end, :])
    n = min(size(data, 2), nrow(events))
    data = data[:, 1:n]
    events_n = events[1:n, :]
    t_extract_prepare = (time_ns() - t_extract0) / 1e9

    # 2) Sort by sort variable
    t_sort0 = time_ns()
    sortvals = sortvalues_from(events_n, sort_col)
    order = sortperm(sortvals)
    data_sorted = data[:, order]
    t_sort = (time_ns() - t_sort0) / 1e9

    # 3) Z-score per timepoint across trials
    t_z0 = time_ns()
    data_z = zscore_timepoints(data_sorted)
    t_zscore = (time_ns() - t_z0) / 1e9

    # 4) Convert to trials x time image
    t_to_img0 = time_ns()
    img_trials_time = Float32.(permutedims(data_z, (2, 1)))
    t_to_trials_time = (time_ns() - t_to_img0) / 1e9

    # 5) Optional low-pass in original resolution
    t_lp0 = time_ns()
    if low_pass_factor > 0
        kernel = gaussian_kernel(low_pass_factor, size(img_trials_time), size(img_trials_time), LOWPASS_KERNEL_SIZE)
        img_trials_time = Float32.(imfilter(img_trials_time, kernel, FILTER_BORDER))
    end
    t_low_pass = (time_ns() - t_lp0) / 1e9

    stage_times = (
        extract_prepare = t_extract_prepare,
        sort = t_sort,
        zscore = t_zscore,
        to_trials_time = t_to_trials_time,
        low_pass = t_low_pass,
    )

    return img_trials_time, stage_times
end

const LS_ID_TO_SYMBOL = Dict(
    "0" => :no_class,
    "1" => :sigmoid,
    "2" => :one_sided_fan,
    "3" => :two_sided_fan,
    "4" => :diverging_bar,
    "5" => :hourglass,
    "6" => :tilted_bar,
)

const LS_NAME_TO_SYMBOL = Dict(
    "no class" => :no_class,
    "sigmoid" => :sigmoid,
    "one sided fan" => :one_sided_fan,
    "two sided fan" => :two_sided_fan,
    "diverging bar" => :diverging_bar,
    "hourglass" => :hourglass,
    "tilted bar" => :tilted_bar,
)

function parse_human_label(v)::Symbol
    s = lowercase(strip(string(v)))
    if haskey(LS_ID_TO_SYMBOL, s)
        return LS_ID_TO_SYMBOL[s]
    end
    s_norm = replace(replace(s, "_" => " "), "-" => " ")
    return get(LS_NAME_TO_SYMBOL, s_norm, Symbol(replace(s_norm, " " => "_")))
end

to_binary_label(lbl::Symbol) = (lbl == :no_class ? 0 : 1)

function parse_image_ref(image_ref::AbstractString)
    m = match(r"erp_(\d+)_ch(\d+)_([A-Za-z0-9_]+)\.png", image_ref)
    m === nothing && error("Could not parse image reference: $image_ref")
    return (
        task_id = parse(Int, m.captures[1]),
        channel = parse(Int, m.captures[2]),
        sort_var = Symbol(m.captures[3]),
    )
end

function resolve_label_row_metadata(row)
    cols = propertynames(row)
    if :channel in cols && :sort_variable in cols && !ismissing(row.channel) && !ismissing(row.sort_variable)
        task_id = (:id in cols && !ismissing(row.id)) ? Int(row.id) : -1
        return (
            task_id = task_id,
            channel = Int(row.channel),
            sort_var = Symbol(String(row.sort_variable)),
        )
    end
    return parse_image_ref(String(row.image))
end



resolve_label_row_metadata (generic function with 1 method)

In [6]:
if !resize_to_model_input
    error("Unscaled model input is not supported by this CNN. It contains Dense(4096 => 128), so input must be 64x64x1.")
end

n = nrow(labels_df)
X_val = Array{Float32}(undef, target_height, target_width, 1, n)
rows = NamedTuple[]

stage_order = [:extract_prepare, :sort, :zscore, :to_trials_time, :low_pass, :resize]
stage_totals = Dict(stage => 0.0 for stage in stage_order)

prep_started = time_ns()

for (i, row) in enumerate(eachrow(labels_df))
    parsed = resolve_label_row_metadata(row)

    img_unscaled, stage_times = process_erp_timed(
        erps,
        events,
        parsed.channel,
        parsed.sort_var,
        time_zero_idx;
        low_pass_factor = LOWPASS_SIGMA,
    )

    t_resize0 = time_ns()
    img_model = Float32.(imresize(img_unscaled, (target_height, target_width)))
    t_resize = (time_ns() - t_resize0) / 1e9

    X_val[:, :, 1, i] = img_model

    human_label = parse_human_label(row.erp_class)
    human_binary = to_binary_label(human_label)

    annotation_id = (:annotation_id in propertynames(row) && !ismissing(row.annotation_id)) ? Int(row.annotation_id) : -1

    push!(rows, (
        idx = i,
        annotation_id = annotation_id,
        task_id = parsed.task_id,
        channel = parsed.channel,
        sort_var = String(parsed.sort_var),
        image = String(row.image),
        human_label = String(human_label),
        human_binary = human_binary,
    ))

    stage_totals[:extract_prepare] += stage_times.extract_prepare
    stage_totals[:sort] += stage_times.sort
    stage_totals[:zscore] += stage_times.zscore
    stage_totals[:to_trials_time] += stage_times.to_trials_time
    stage_totals[:low_pass] += stage_times.low_pass
    stage_totals[:resize] += t_resize
end

prep_total_s = (time_ns() - prep_started) / 1e9
stages_total_s = sum(values(stage_totals))
overhead_s = max(prep_total_s - stages_total_s, 0.0)

timing_rows = NamedTuple[]
calls = n
for stage in stage_order
    total_s = stage_totals[stage]
    avg_ms = calls == 0 ? 0.0 : 1000.0 * total_s / calls
    pct = stages_total_s == 0 ? 0.0 : 100.0 * total_s / stages_total_s
    push!(timing_rows, (
        stage = String(stage),
        calls = calls,
        total_s = round(total_s, digits = 4),
        avg_ms = round(avg_ms, digits = 3),
        pct_time = round(pct, digits = 2),
    ))
end

prep_timing_df = DataFrame(timing_rows)
val_df = DataFrame(rows)

println("Validation tensor size: ", size(X_val), " (H, W, C, N)")
println("Human binary counts: no_pattern=", count(==(0), val_df.human_binary), " | pattern=", count(==(1), val_df.human_binary))
println(@sprintf("Preprocessing wall time: %.3f s | stage sum: %.3f s | overhead: %.3f s", prep_total_s, stages_total_s, overhead_s))
println("Preprocessing timing (manual order):")
show(prep_timing_df, allrows = true, allcols = true)
println()



Validation tensor size: (64, 64, 1, 500) (H, W, C, N)
Human binary counts: no_pattern=451 | pattern=49
Preprocessing wall time: 9.115 s | stage sum: 8.470 s | overhead: 0.645 s
Preprocessing timing (manual order):
6×5 DataFrame
 Row │ stage            calls  total_s  avg_ms   pct_time 
     │ String           Int64  Float64  Float64  Float64  
─────┼────────────────────────────────────────────────────
   1 │ extract_prepare    500   3.5464    7.093     41.87
   2 │ sort               500   0.4102    0.82       4.84
   3 │ zscore             500   1.4677    2.935     17.33
   4 │ to_trials_time     500   0.7345    1.469      8.67
   5 │ low_pass           500   2.1181    4.236     25.01
   6 │ resize             500   0.1928    0.386      2.28


In [7]:
# Input summary table (no plot)
stats_rows = NamedTuple[]
for i in 1:size(X_val, 4)
    img = X_val[:, :, 1, i]
    push!(stats_rows, (
        idx = i,
        channel = Int(val_df.channel[i]),
        sort_var = String(val_df.sort_var[i]),
        human_label = String(val_df.human_label[i]),
        min_val = Float32(minimum(img)),
        max_val = Float32(maximum(img)),
        mean_val = Float32(mean(img)),
        std_val = Float32(std(img)),
    ))
end

input_stats_df = DataFrame(stats_rows)
input_stats_df.min_val = round.(input_stats_df.min_val, digits = 4)
input_stats_df.max_val = round.(input_stats_df.max_val, digits = 4)
input_stats_df.mean_val = round.(input_stats_df.mean_val, digits = 4)
input_stats_df.std_val = round.(input_stats_df.std_val, digits = 4)

println("Model input summary (first 20 rows):")
first(input_stats_df, min(20, nrow(input_stats_df)))



Model input summary (first 20 rows):


Row,idx,channel,sort_var,human_label,min_val,max_val,mean_val,std_val
,Int64,Int64,String,String,Float32,Float32,Float32,Float32
1,1,1,duration,no_class,-0.7147,0.6928,-0.0192,0.1984
2,2,2,sac_amplitude,no_class,-0.7349,0.5959,0.0081,0.1847
3,3,3,fix_avgpos_x,no_class,-0.7662,1.2025,-0.0095,0.1905
4,4,4,fix_avgpupilsize,no_class,-0.6025,0.5851,0.0052,0.174
5,5,5,fix_type,no_class,-0.7758,0.6591,-0.004,0.212
6,6,6,latency,no_class,-0.7371,1.5469,-0.0035,0.3057
7,7,8,duration,sigmoid,-0.5649,0.7151,-0.0072,0.1941
8,8,9,sac_amplitude,no_class,-0.566,0.5118,-0.0014,0.1679
9,9,10,fix_avgpos_x,no_class,-0.5792,0.62,0.0125,0.1895


In [8]:
probs = Array(cpu(model(X_val)))
@assert size(probs, 1) == 2 "Expected 2 classes in the model"
@assert size(probs, 2) == nrow(val_df) "Prediction count mismatch"

pred_binary = onecold(probs, 0:1)
pred_label = [String(model_pattern_names[p + 1]) for p in pred_binary]

val_df.pred_binary = pred_binary
val_df.pred_label = pred_label
val_df.prob_no_pattern = vec(probs[1, :])
val_df.prob_pattern = vec(probs[2, :])
val_df.correct = val_df.pred_binary .== val_df.human_binary

accuracy = mean(Float64.(val_df.correct))

function confusion_binary(y_true::Vector{Int}, y_pred::Vector{Int})
    cm = zeros(Int, 2, 2)
    for (t, p) in zip(y_true, y_pred)
        cm[t + 1, p + 1] += 1
    end
    return cm
end

cm = confusion_binary(Vector{Int}(val_df.human_binary), Vector{Int}(val_df.pred_binary))

println("Binary accuracy (manual vs model): ", @sprintf("%.4f", accuracy))



Binary accuracy (manual vs model): 0.2840


In [9]:
# Detailed table
sort!(val_df, [:correct, :prob_pattern], rev = [false, true])
first(val_df, min(20, nrow(val_df)))


Row,idx,annotation_id,task_id,channel,sort_var,image,human_label,human_binary,pred_binary,pred_label,prob_no_pattern,prob_pattern,correct
,Int64,Int64,Int64,Int64,String,String,String,Int64,Int64,String,Float32,Float32,Bool
1,3,15,115,3,fix_avgpos_x,/data/local-files/?d=label_studio_data_unlabelled/images/erp_003_ch003_fix_avgpos_x.png,no_class,0,1,pattern,2.44695e-13,1.0,false
2,5,17,117,5,fix_type,/data/local-files/?d=label_studio_data_unlabelled/images/erp_005_ch005_fix_type.png,no_class,0,1,pattern,8.53608e-12,1.0,false
3,6,18,118,6,latency,/data/local-files/?d=label_studio_data_unlabelled/images/erp_006_ch006_latency.png,no_class,0,1,pattern,5.03844e-23,1.0,false
4,9,21,121,10,fix_avgpos_x,/data/local-files/?d=label_studio_data_unlabelled/images/erp_009_ch010_fix_avgpos_x.png,no_class,0,1,pattern,1.49227e-13,1.0,false
5,10,22,122,11,fix_avgpupilsize,/data/local-files/?d=label_studio_data_unlabelled/images/erp_010_ch011_fix_avgpupilsize.png,no_class,0,1,pattern,6.12349e-12,1.0,false
6,13,25,125,15,duration,/data/local-files/?d=label_studio_data_unlabelled/images/erp_013_ch015_duration.png,no_class,0,1,pattern,0.0,1.0,false
7,14,26,126,16,sac_amplitude,/data/local-files/?d=label_studio_data_unlabelled/images/erp_014_ch016_sac_amplitude.png,no_class,0,1,pattern,1.13856e-8,1.0,false
8,18,30,130,20,latency,/data/local-files/?d=label_studio_data_unlabelled/images/erp_018_ch020_latency.png,no_class,0,1,pattern,1.9574e-12,1.0,false
9,20,32,132,23,sac_amplitude,/data/local-files/?d=label_studio_data_unlabelled/images/erp_020_ch023_sac_amplitude.png,no_class,0,1,pattern,6.5886e-9,1.0,false


In [10]:
# Pretty confusion matrix + metrics tables

tn = cm[1, 1]
fp = cm[1, 2]
fn = cm[2, 1]
tp = cm[2, 2]

row_totals = [tn + fp, fn + tp]
col_totals = [tn + fn, fp + tp]
n_total = sum(cm)

function fmt_cell_with_row_pct(count::Int, row_total::Int)
    pct = row_total == 0 ? 0.0 : 100.0 * count / row_total
    return @sprintf("%d (%.1f%%)", count, pct)
end

confusion_table = [
    "no_pattern"  fmt_cell_with_row_pct(tn, row_totals[1])  fmt_cell_with_row_pct(fp, row_totals[1])  string(row_totals[1]);
    "pattern"     fmt_cell_with_row_pct(fn, row_totals[2])  fmt_cell_with_row_pct(tp, row_totals[2])  string(row_totals[2]);
    "Total"       string(col_totals[1])                     string(col_totals[2])                     string(n_total)
]

println("Confusion Matrix (rows=True, cols=Pred):")
pretty_table(
    confusion_table;
    column_labels = [raw"True\Pred", "no_pattern", "pattern", "Total"],
    alignment = [:l, :r, :r, :r],
)

precision = (tp + fp) == 0 ? 0.0 : tp / (tp + fp)
recall = (tp + fn) == 0 ? 0.0 : tp / (tp + fn)
f1 = (precision + recall) == 0 ? 0.0 : 2 * precision * recall / (precision + recall)
specificity = (tn + fp) == 0 ? 0.0 : tn / (tn + fp)

metrics_table = [
    "accuracy"                  @sprintf("%.3f", accuracy);
    "precision_pattern"         @sprintf("%.3f", precision);
    "recall_pattern"            @sprintf("%.3f", recall);
    "f1_pattern"                @sprintf("%.3f", f1);
    "specificity_no_pattern"    @sprintf("%.3f", specificity)
]

println("\nMetrics:")
pretty_table(
    metrics_table;
    column_labels = ["Metric", "Value"],
    alignment = [:l, :r],
)





Confusion Matrix (rows=True, cols=Pred):
┌────────────┬─────────────┬─────────────┬───────┐
│ True\\Pred │  no_pattern │     pattern │ Total │
├────────────┼─────────────┼─────────────┼───────┤
│ no_pattern │ 104 (23.1%) │ 347 (76.9%) │   451 │
│ pattern    │  11 (22.4%) │  38 (77.6%) │    49 │
│ Total      │         115 │         385 │   500 │
└────────────┴─────────────┴─────────────┴───────┘

Metrics:
┌────────────────────────┬───────┐
│ Metric                 │ Value │
├────────────────────────┼───────┤
│ accuracy               │ 0.284 │
│ precision_pattern      │ 0.099 │
│ recall_pattern         │ 0.776 │
│ f1_pattern             │ 0.175 │
│ specificity_no_pattern │ 0.231 │
└────────────────────────┴───────┘


In [11]:
# Class distribution across all manually labeled ERP images

@assert @isdefined(val_df) "val_df is not defined. Run the preprocessing/prediction cells first."

class_order = [
    :sigmoid,
    :one_sided_fan,
    :two_sided_fan,
    :diverging_bar,
    :hourglass,
    :tilted_bar,
    :no_class,
]

class_display = Dict(
    :sigmoid => "Sigmoid",
    :one_sided_fan => "One Sided Fan",
    :two_sided_fan => "Two Sided Fan",
    :diverging_bar => "Diverging Bar",
    :hourglass => "Hourglass",
    :tilted_bar => "Tilted Bar",
    :no_class => "No Class",
)

labels_sym = Symbol.(val_df.human_label)
n_total = length(labels_sym)

dist_rows = Matrix{String}(undef, length(class_order) + 1, 3)
for (i, cls) in enumerate(class_order)
    c = count(==(cls), labels_sym)
    pct = n_total == 0 ? 0.0 : 100.0 * c / n_total
    dist_rows[i, 1] = class_display[cls]
    dist_rows[i, 2] = string(c)
    dist_rows[i, 3] = @sprintf("%.1f%%", pct)
end

dist_rows[end, 1] = "Total"
dist_rows[end, 2] = string(n_total)
dist_rows[end, 3] = "100.0%"

println("Class distribution across manually labeled ERP images:")
pretty_table(
    dist_rows;
    column_labels = ["Class", "Count", "Share"],
    alignment = [:l, :r, :r],
)

pattern_total = n_total - count(==(:no_class), labels_sym)
pattern_pct = n_total == 0 ? 0.0 : 100.0 * pattern_total / n_total
no_class_total = n_total - pattern_total
no_class_pct = n_total == 0 ? 0.0 : 100.0 * no_class_total / n_total

pattern_summary = [
    "Any pattern (classes 1-6)" string(pattern_total) @sprintf("%.1f%%", pattern_pct);
    "No Class"                 string(no_class_total) @sprintf("%.1f%%", no_class_pct)
]

println("\nPattern vs No Class summary:")
pretty_table(
    pattern_summary;
    column_labels = ["Group", "Count", "Share"],
    alignment = [:l, :r, :r],
)

# Grouped by sort variable: where do patterns and no class come from
sort_rows = NamedTuple[]
for g in groupby(val_df, :sort_var)
    sv = String(first(g.sort_var))
    n_sv = nrow(g)
    no_class_sv = count(==("no_class"), g.human_label)
    pattern_sv = n_sv - no_class_sv
    pattern_pct_sv = n_sv == 0 ? 0.0 : 100.0 * pattern_sv / n_sv
    no_class_pct_sv = n_sv == 0 ? 0.0 : 100.0 * no_class_sv / n_sv
    push!(sort_rows, (
        sort_var = sv,
        total = n_sv,
        pattern_count = pattern_sv,
        pattern_share = pattern_pct_sv,
        no_class_count = no_class_sv,
        no_class_share = no_class_pct_sv,
    ))
end

sort_df = DataFrame(sort_rows)
sort!(sort_df, :sort_var)

sort_table = Matrix{String}(undef, nrow(sort_df), 6)
for i in 1:nrow(sort_df)
    sort_table[i, 1] = sort_df.sort_var[i]
    sort_table[i, 2] = string(sort_df.total[i])
    sort_table[i, 3] = string(sort_df.pattern_count[i])
    sort_table[i, 4] = @sprintf("%.1f%%", sort_df.pattern_share[i])
    sort_table[i, 5] = string(sort_df.no_class_count[i])
    sort_table[i, 6] = @sprintf("%.1f%%", sort_df.no_class_share[i])
end

println("\nDistribution by sort variable:")
pretty_table(
    sort_table;
    column_labels = ["Sort Variable", "Total", "Pattern", "Pattern %", "No Class", "No Class %"],
    alignment = [:l, :r, :r, :r, :r, :r],
)



Class distribution across manually labeled ERP images:
┌───────────────┬───────┬────────┐
│ Class         │ Count │  Share │
├───────────────┼───────┼────────┤
│ Sigmoid       │    36 │   7.2% │
│ One Sided Fan │     0 │   0.0% │
│ Two Sided Fan │     2 │   0.4% │
│ Diverging Bar │     6 │   1.2% │
│ Hourglass     │     3 │   0.6% │
│ Tilted Bar    │     2 │   0.4% │
│ No Class      │   451 │  90.2% │
│ Total         │   500 │ 100.0% │
└───────────────┴───────┴────────┘

Pattern vs No Class summary:
┌───────────────────────────┬───────┬───────┐
│ Group                     │ Count │ Share │
├───────────────────────────┼───────┼───────┤
│ Any pattern (classes 1-6) │    49 │  9.8% │
│ No Class                  │   451 │ 90.2% │
└───────────────────────────┴───────┴───────┘

Distribution by sort variable:
┌──────────────────┬───────┬─────────┬───────────┬──────────┬────────────┐
│ Sort Variable    │ Total │ Pattern │ Pattern % │ No Class │ No Class % │
├──────────────────┼───────┼─────────